# 05 — Exploratory native-output reference agreement

**Place this notebook in the existing suite/ folder. It reads finished results; no segmentation is rerun.**

This version permits the selected SMURF variant's gene-wise count excess for an explicitly exploratory comparison. Every other source-integrity and frozen-reference check still applies. Counts, images, masks, input files and recorded runtimes remain unchanged. Each execution writes a new report directory.

Defaults: all baseline bundles, repeat 1, three methods, SMURF count-excess continuation enabled. The supplied finished notebooks report eight ovarian bundles per method; the checks below verify their current files. Embedded helpers avoid replacing the installed benchmark modules.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display
SUITE_DIR = next((p for parent in [Path.cwd(), *Path.cwd().parents]
                  for p in [parent, parent / "suite"] if (p / "benchmark_session.py").is_file()), None)
if SUITE_DIR is None:
    raise FileNotFoundError("Open this notebook from the unpacked public benchmark (or its suite/ directory).")
sys.path.insert(0, str(SUITE_DIR))
from benchmark_session import start
from p06212_benchmark.common import load_config, output_root, write_json, read_json
CONFIG_PATH, config, ROOT = start()

## 1 · Select completed runs
The exception is restricted to SMURF count excess after all other checks pass. It is not a general ignore-errors setting. Every observed excess is reported; no size threshold silently calls it conservation.

In [ ]:
from datetime import datetime, timezone
import numpy as np
from p06212_benchmark.common import file_sha256
from p06212_benchmark.evaluation import evaluate_h5ad
from p06212_benchmark.evaluation_report import collect_evaluations, plot_benchmark_report

SCOPE = "all"  # "all" or "pilot"
REPEAT_IDS = [1]
ALLOW_SMURF_GENE_EXCESS = True  # requested exploratory continuation; native counts are unchanged

manifest = pd.read_csv(ROOT / "condition_manifest.csv", dtype={"bundle_id":str,"condition_id":str})
lookup = manifest.set_index(["bundle_id","condition_id"], verify_integrity=True)
smurf_method = "smurf_" + config["methods"]["smurf"]["variant"]
methods = ["bin2cell", smurf_method, "proseg"]
base_ids = list(dict.fromkeys(manifest.loc[manifest.condition_id.eq("baseline"),"bundle_id"]))
if not base_ids or SCOPE not in ("all", "pilot"):
    raise ValueError("Unknown scope or no baseline bundles in the manifest.")
bundle_ids = base_ids if SCOPE == "all" else base_ids[:1]
REPORT = ROOT / "reports" / ("exploratory_native_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
REPORT.mkdir(parents=True, exist_ok=False)
print("Specimen:", next(iter(config["samples"])))
print("Bundles:", bundle_ids, "| Methods:", methods, "| Repeats:", REPEAT_IDS)
print("Report:", REPORT)

### Embedded audit helpers
The H5AD audit uses bounded row slices. It independently checks gene totals against source bins, the saved gene audit and run summary. Completed runner/launcher records, positive recorded time, input and mask hashes, finite nonnegative integer counts, and a seed audit remain required. All checks after the former excess-count exception still run.

In [ ]:
"""Embedded exploratory audit: retain native counts and every non-count check."""
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from p06212_benchmark.common import read_json, file_sha256


def _native_gene_totals(path):
    import anndata as ad
    obj = ad.read_h5ad(path, backed="r")
    try:
        names = pd.Index(obj.var_names.astype(str))
        if not names.is_unique:
            raise ValueError("Duplicate gene axis")
        total = np.zeros(len(names), dtype=np.int64)
        step = max(1, min(50_000, 32 * 2**20 // max(1, 8 * len(names))))
        for first in range(0, obj.n_obs, step):
            block = obj.X[first:first + step, :]
            values = block.data if sparse.issparse(block) else np.asarray(block)
            if (not np.isfinite(values).all() or np.any(values < 0)
                    or np.any(values != np.rint(values))):
                raise ValueError("Expected finite nonnegative integer native counts")
            total += np.asarray(block.sum(axis=0, dtype=np.int64)).ravel()
        return names, total
    finally:
        obj.file.close()


def audit_native_outputs(root, lookup, bundle_ids, methods, repeat_ids, *, allowed_excess_methods=()):
    """Allow only an explicitly named method's gene-wise excess after all other checks.

    Returns audit rows, exactly the accepted native paths, and per-gene excess details.
    An allowed excess is not relabeled as count conservation. No data are modified.
    """
    root = Path(root)
    if not bundle_ids or not methods or not repeat_ids:
        raise ValueError("Choose a nonempty set of bundles, methods and repeats.")
    if any(type(i) is not int or i < 1 for i in repeat_ids):
        raise ValueError("Repeat IDs must be positive integers.")
    if any(len(set(x)) != len(x) for x in (bundle_ids, methods, repeat_ids)):
        raise ValueError("Duplicate requested IDs")
    allowed = set(allowed_excess_methods)
    if allowed - set(methods) or any(not m.startswith("smurf_") for m in allowed):
        raise ValueError("This notebook's exception is restricted to selected SMURF variants.")
    rows, accepted, details = [], [], []
    for bundle in bundle_ids:
        meta_row = lookup.loc[(bundle, "baseline")]
        inp = Path(meta_row.input_dir)
        stored = read_json(inp / "metadata.json")
        mask_hash = file_sha256(inp / stored.get("seed_filename", "seeds.npy"))
        meta_hash = file_sha256(inp / "metadata.json")
        genes, available = _native_gene_totals(inp / stored.get("bins_filename", "bins.h5ad"))
        source_total = int(available.sum())
        if source_total != int(stored["n_input_molecules"]):
            raise ValueError(f"{bundle}: current public counts differ from input metadata.")
        for method in methods:
            for repeat in repeat_ids:
                dest = root / "results" / bundle / "baseline" / method / f"repeat_{repeat:02d}"
                prediction = dest / "counts.h5ad"
                summary_path = dest / "run_summary.json"
                launcher_path = dest.with_name(dest.name + "_launcher.json")
                row = dict(bundle_id=bundle, condition="baseline", method=method, repeat=repeat,
                           status="blocked", reason="", integrity_checks_passed=False,
                           source_bundle_has_gene_excess=False, accepted_for_exploratory=False,
                           run_summary_path=str(summary_path), launcher_path=str(launcher_path),
                           predicted_counts_path=str(prediction), input_dir=str(inp))
                try:
                    summary, launcher = read_json(summary_path), read_json(launcher_path)
                    elapsed = float(summary["total_runner_seconds"])
                    if (summary.get("method") != method or summary.get("status") != "completed"
                            or launcher.get("returncode") != 0 or not np.isfinite(elapsed) or elapsed <= 0):
                        raise ValueError("Need a completed runner, successful launcher and positive recorded runtime.")
                    if (Path(summary["input_dir"]).resolve() != inp.resolve()
                            or summary.get("seed_sha256") != mask_hash
                            or summary.get("metadata_sha256") != meta_hash):
                        raise ValueError("Input path, metadata hash or seed-mask hash differs from the completed run.")
                    saved = pd.read_csv(dest / "gene_count_audit.csv", dtype={"gene": str}).set_index("gene", verify_integrity=True)
                    outgenes, allocated = _native_gene_totals(prediction)
                    if not genes.equals(outgenes) or set(saved.index) != set(genes):
                        raise ValueError("Input, output or audit gene axes differ.")
                    if (not np.array_equal(saved.loc[genes, "input_counts"].to_numpy(), available)
                            or not np.array_equal(saved.loc[genes, "output_counts"].to_numpy(), allocated)):
                        raise ValueError("Current native gene totals disagree with the saved gene-count audit.")
                    extra = np.maximum(allocated - available, 0)
                    n_extra = int(extra.sum())
                    n_genes = int((extra > 0).sum())
                    out_total = int(allocated.sum())
                    row.update(input_counts=source_total, output_counts=out_total,
                               gene_excess_counts=n_extra, n_excess_genes=n_genes,
                               max_gene_excess_counts=int(extra.max()) if len(extra) else 0,
                               gene_excess_fraction_of_input=n_extra / source_total if source_total else np.nan,
                               output_fraction_of_input=out_total / source_total if source_total else np.nan,
                               source_bundle_has_gene_excess=n_extra > 0,
                               seed_sha256=mask_hash, metadata_sha256=meta_hash, total_runner_seconds=elapsed,
                               launcher_wall_seconds=launcher.get("wall_seconds_including_interpreter"))
                    for j in np.flatnonzero(extra):
                        details.append(dict(bundle_id=bundle, condition="baseline", method=method, repeat=repeat,
                                            gene=str(genes[j]), input_counts=int(available[j]),
                                            output_counts=int(allocated[j]), excess_counts=int(extra[j]),
                                            excess_fraction_of_gene_input=float(extra[j] / available[j]) if available[j] else np.nan))
                    # These were AFTER the old count-excess raise. They must still run.
                    if (int(summary["input_counts"]) != source_total
                            or int(summary["output_counts"]) != out_total):
                        raise ValueError("Current native count totals disagree with run_summary.json.")
                    if not (dest / "seed_audit.csv").is_file():
                        raise FileNotFoundError("Missing seed_audit.csv")
                    row["integrity_checks_passed"] = True
                    if n_extra and method not in allowed:
                        raise ValueError("Gene-count excess is outside the selected SMURF exploratory exception.")
                    row["status"] = "verified_with_gene_excess" if n_extra else "verified"
                    row["reason"] = "Native gene-count excess retained for exploratory evaluation." if n_extra else ""
                    row["accepted_for_exploratory"] = True
                    accepted.append(prediction)
                except (OSError, TypeError, ValueError, KeyError, IndexError, OverflowError) as exc:
                    row["reason"] = f"{type(exc).__name__}: {exc}"
                rows.append(row)
    gene_columns = ["bundle_id", "condition", "method", "repeat", "gene", "input_counts", "output_counts",
                    "excess_counts", "excess_fraction_of_gene_input"]
    return pd.DataFrame(rows), accepted, pd.DataFrame(details, columns=gene_columns)


def attach_source_diagnostics(frame, audit):
    """Whole-bundle provenance only; do not infer which core or cell has extra counts."""
    if frame.empty:
        return frame.copy()
    keys = ["bundle_id", "condition", "method", "replicate"]
    if not set(keys).issubset(frame):
        raise ValueError(f"Report table is missing source identity fields: {sorted(set(keys) - set(frame))}")
    diagnostic = audit.rename(columns={"repeat": "replicate", "status": "source_audit_status",
                                      "gene_excess_counts": "source_bundle_gene_excess_counts",
                                      "n_excess_genes": "source_bundle_n_excess_genes",
                                      "gene_excess_fraction_of_input": "source_bundle_gene_excess_fraction_of_input"})
    cols = keys + ["source_audit_status", "source_bundle_has_gene_excess", "source_bundle_gene_excess_counts",
                   "source_bundle_n_excess_genes", "source_bundle_gene_excess_fraction_of_input"]
    result = frame.merge(diagnostic[cols], on=keys, how="left", validate="many_to_one", indicator="_source_join")
    if not result["_source_join"].eq("both").all():
        raise ValueError("At least one result row does not match an audited run source.")
    result = result.drop(columns="_source_join")
    result["source_bundle_has_gene_excess"] = result["source_bundle_has_gene_excess"].astype(bool)
    result["comparison_status"] = "exploratory_native_output"
    return result


def check_complete_core_sets(cores, selection, bundle_ids, methods, repeat_ids):
    keys = ["bundle_id", "sample_id", "core_id"]
    chosen = selection.loc[selection.bundle_id.isin(bundle_ids)]
    expected = set(map(tuple, chosen[keys].astype(str).itertuples(index=False, name=None)))
    expected_runs = {(m, r) for m in methods for r in repeat_ids}
    actual_runs = set(zip(cores.method, cores.replicate))
    if actual_runs != expected_runs or not expected:
        raise ValueError("Not all requested methods/repeats have evaluable selected cores.")
    for (m, r), frame in cores.groupby(["method", "replicate"], observed=True):
        observed = set(map(tuple, frame[keys].astype(str).itertuples(index=False, name=None)))
        if observed != expected or len(frame) != len(expected):
            raise ValueError(f"{m}, repeat {r}: missing, duplicated or extra selected core rows.")

## 2 · Audit selected sources and quantify excess

- **verified:** integrity checks passed and no gene-wise excess was detected.
- **verified_with_gene_excess:** every other check passed; native SMURF excess is included and flagged.
- **blocked:** a different check failed. No method is silently omitted.

Gene excess means the sum of positive output-minus-input count differences per gene across the full inference bundle, including context. This is not a count of new gene features, a cell-level contamination measure, or a localization of extra molecules. Losses for other genes can coexist with excess.

In [ ]:
audit, result_files, excess_genes = audit_native_outputs(
    ROOT, lookup, bundle_ids, methods, REPEAT_IDS,
    allowed_excess_methods=[smurf_method] if ALLOW_SMURF_GENE_EXCESS else [],
)
audit.to_csv(REPORT / "fresh_run_audit.csv", index=False)
excess_genes.to_csv(REPORT / "gene_excess_detail.csv", index=False)
write_json(REPORT / "comparison_policy.json", {
    "comparison_status":"exploratory_native_output", "scope":SCOPE,
    "bundles":bundle_ids,"methods":methods,"repeat_ids":REPEAT_IDS,
    "allow_smurf_gene_excess":ALLOW_SMURF_GENE_EXCESS,
    "native_counts_modified":False,"segmentation_rerun":False,
    "exception":"Only SMURF gene-wise excess after every other source check passes.",
    "diagnostic_unit":"Source bundle, method, baseline condition, repeat. Core flags do not localize excess.",
    "interpretation":"Gene-count reference agreement conditional on frozen matched nuclei, not molecule-identity accuracy.",
})
columns=["bundle_id","method","repeat","status","input_counts","output_counts",
         "gene_excess_counts","n_excess_genes","gene_excess_fraction_of_input","reason"]
display(audit[[c for c in columns if c in audit]])
if not excess_genes.empty:
    display(excess_genes)
expected_n = len(bundle_ids) * len(methods) * len(REPEAT_IDS)
if len(audit) != expected_n or not audit.accepted_for_exploratory.all() or len(result_files) != expected_n:
    raise ValueError(f"A nonpermitted integrity failure remains. See {REPORT / 'fresh_run_audit.csv'}. No incomplete method comparison was scored.")

# Preserve the selected records in this report; do not scan unrelated or stale jobs.
for item in audit.itertuples(index=False):
    folder = REPORT / "selected_run_records" / item.bundle_id / item.condition / item.method / f"repeat_{item.repeat:02d}"
    write_json(folder / "run_summary.json", read_json(item.run_summary_path))
    write_json(folder / "launcher.json", read_json(item.launcher_path))
print("Included sources:", len(result_files), "| Sources with excess:", int(audit.source_bundle_has_gene_excess.sum()))
print("The audit is complete. Proceed to unchanged-count evaluation.")

## 3 · Score the shared frozen reference identities
The original evaluator and metrics are reused. Missing predicted cells remain zero. Counts are never clipped, redistributed, rounded or normalized here. Mask/identity hashes and reference coverage are still checked.

In [ ]:
failures=[]
for job_index, prediction_path in enumerate(result_files, start=1):
    bundle, condition, method, repeat_name = prediction_path.relative_to(ROOT / "results").parts[:4]
    print(f"[{job_index}/{len(result_files)}] Scoring {bundle} / {method} / {repeat_name}", flush=True)
    row = lookup.loc[(bundle,condition)]
    reference_dir = Path(row.reference_dir)
    mapping_review = reference_dir / "mapping_review.json"
    if not mapping_review.exists():
        raise FileNotFoundError(f"Review/freeze the baseline seed map first: {reference_dir}")
    proof = read_json(mapping_review)
    if not proof.get("frozen") or file_sha256(reference_dir / "seed_map.csv") != proof["seed_map_sha256"]:
        raise ValueError("The frozen identity map changed after review.")
    input_meta = read_json(Path(row.input_dir) / "metadata.json")
    baseline_hash = input_meta.get("baseline_seeds_sha256", input_meta.get("seeds_sha256"))
    if baseline_hash != proof["baseline_seeds_sha256"]:
        raise ValueError("These method inputs do not originate from the reviewed baseline mask.")
    frozen = pd.read_csv(reference_dir / "seed_map.csv",
        dtype={"seed_id":str,"reference_cell_id":str,"sample_id":str,"component_id":str,"core_id":str})
    frozen["bundle_id"] = bundle
    frozen["dx_um"] = float(row.dx_um); frozen["dy_um"] = float(row.dy_um)
    frozen["added_offset_um"] = float(np.hypot(row.dx_um,row.dy_um))
    destination = REPORT / "evaluation" / bundle / condition / method / repeat_name
    try:
        result = evaluate_h5ad(prediction_path, reference_dir / "counts.h5ad", frozen,
                               method=method, condition=condition, replicate=int(repeat_name.split("_")[-1]))
        diagnostic = audit.loc[audit.bundle_id.eq(bundle) & audit.condition.eq(condition)
                               & audit.method.eq(method) & audit["repeat"].eq(int(repeat_name.split("_")[-1]))]
        if len(diagnostic) != 1:
            raise ValueError("Expected exactly one selected source audit row.")
        d = diagnostic.iloc[0]
        result.audit.update(bundle_id=bundle, comparison_status="exploratory_native_output",
                            source_audit_status=str(d["status"]),
                            source_bundle_has_gene_excess=bool(d["source_bundle_has_gene_excess"]),
                            source_bundle_gene_excess_counts=int(d["gene_excess_counts"]),
                            source_bundle_n_excess_genes=int(d["n_excess_genes"]))
        result.save(destination)
        print(bundle,condition,method,repeat_name,"scored:",len(result.cell_metrics))
    except Exception as exc:
        failures.append({"prediction":str(prediction_path),"error":f"{type(exc).__name__}: {exc}"})
write_json(REPORT / "evaluation_failures.json", failures)
if failures:
    display(pd.DataFrame(failures))
    raise RuntimeError("Resolve evaluation failures before interpreting a method comparison.")

## 4 · Export metrics, coverage and original runtimes

Source flags join by bundle, method, condition and repeat. Whole-bundle diagnostics repeated on core rows are provenance, not quantities to sum across cores; use fresh_run_audit.csv for source totals. Recorded method runtimes come from the completed runs, not this evaluation notebook.

In [ ]:
tables = collect_evaluations(REPORT / "evaluation", REPORT / "selected_run_records")
run_keys = ["bundle_id","condition","method","replicate"]
run_identity = audit.rename(columns={"repeat":"replicate"})[run_keys + ["run_summary_path"]]
tables.runs = tables.runs.merge(run_identity, on=run_keys, how="left", validate="one_to_one")
if len(tables.runs) != len(audit) or tables.runs.run_summary_path.isna().any():
    raise ValueError("Recorded runtime sources do not match the complete scored run set.")
tables.runs = tables.runs.rename(columns={"source_file":"report_record_file"})
tables.runs["source_file"] = tables.runs.run_summary_path

selection = pd.read_csv(ROOT / "review" / "selected_cores.csv", dtype={"sample_id":str,"bundle_id":str,"core_id":str})
check_complete_core_sets(tables.cores, selection, bundle_ids, methods, REPEAT_IDS)
tables.cores = attach_source_diagnostics(tables.cores, audit)
tables.celltypes = attach_source_diagnostics(tables.celltypes, audit)
tables.coverage = attach_source_diagnostics(tables.coverage, audit)
tables.runs = attach_source_diagnostics(tables.runs, audit)
for filename, frame in [("all_core_metrics",tables.cores),("all_celltype_metrics",tables.celltypes),
                        ("all_reference_seed_coverage",tables.coverage),("all_runtime_audits",tables.runs),
                        ("all_evaluation_audits",tables.audits)]:
    frame.to_csv(REPORT / (filename + ".csv"), index=False)
write_json(REPORT / "collection_notes.json", tables.issues)
display(tables.cores[["bundle_id","core_id","method","replicate","normalized_l1_gene_count_error",
                     "median_profile_cosine","gene_count_recovery","source_bundle_has_gene_excess"]])

# Secondary sensitivity table: remove each affected bundle for every method/repeat together.
# The main tables and plots retain ALL selected bundles.
affected_bundles = set(audit.loc[audit.source_bundle_has_gene_excess,"bundle_id"])
sensitivity = tables.cores.loc[~tables.cores.bundle_id.isin(affected_bundles)].copy()
sensitivity.to_csv(REPORT / "core_metrics_without_any_flagged_bundle.csv", index=False)
write_json(REPORT / "sensitivity_subset.json", {
    "excluded_bundles_for_every_method":sorted(affected_bundles),
    "retained_bundles":sorted(set(bundle_ids) - affected_bundles),
    "is_primary_comparison":False,
    "note":"Descriptive sensitivity subset only; removing tissue regions changes the study population.",
})
print("All selected native outputs scored; sensitivity table has", len(sensitivity), "core/method/repeat rows.")

## 5 · Plot native-output agreement with source flags
Blue = bin2cell; orange = SMURF lite; teal = Proseg. Red hollow triangles flag cores whose method/bundle has excess in any included repeat. They do not mean every flagged core or cell contains extra counts.

In [ ]:
"""Standalone notebook-embeddable plot of unchanged native reference agreement."""

def plot_exploratory_agreement(tables, audit, output_path, title):
    """Save 2x2 baseline reference-agreement PNG and PDF; no source counts are changed.

    One dot is a core's mean metric over the requested technical repeats. Triangles
    flag all cores from a method/bundle for which any included repeat has gene-wise
    count excess. They do not locate excess molecules inside those cores.
    """
    from pathlib import Path
    import textwrap
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from p06212_benchmark.evaluation_report import core_means

    metrics = ["normalized_l1_gene_count_error", "gene_count_recovery",
               "gene_count_precision", "output_seed_retention_fraction"]
    required = ["bundle_id", "sample_id", "core_id", "method", "condition", "replicate", *metrics]
    absent = set(required) - set(tables.cores.columns)
    if absent:
        raise ValueError(f"Core table lacks required plotting fields: {sorted(absent)}")
    src = tables.cores.loc[tables.cores["condition"].eq("baseline")].copy()
    if src.empty:
        raise ValueError("No baseline core measurements to plot.")
    if audit.empty or not audit["status"].isin(["verified", "verified_with_gene_excess"]).all():
        raise ValueError("Complete all non-excess run integrity checks before plotting.")
    source_keys = ["bundle_id", "method", "condition", "replicate"]
    diag = audit.rename(columns={"repeat": "replicate"}).copy()
    if diag.duplicated(source_keys).any():
        raise ValueError("Duplicate source audit rows.")
    diag["_audit_excess"] = diag["status"].eq("verified_with_gene_excess")
    src = src.merge(diag[source_keys + ["_audit_excess"]], on=source_keys,
                    how="left", validate="many_to_one")
    if src["_audit_excess"].isna().any():
        raise ValueError("Some plotted cores have no selected-source audit.")
    if "source_bundle_has_gene_excess" in src:
        supplied = src["source_bundle_has_gene_excess"]
        if not pd.api.types.is_bool_dtype(supplied.dtype):
            supplied = supplied.map({True: True, False: False, "True": True, "False": False,
                                     "true": True, "false": False, 1: True, 0: False})
        if supplied.isna().any() or not supplied.astype(bool).equals(src["_audit_excess"].astype(bool)):
            raise ValueError("Core source-excess flags disagree with the verified source audit.")

    # Only original score/metadata fields enter technical-repeat averaging.
    # In particular, bundle-level count totals and flags are not averaged as scores.
    metadata = [c for c in ["component_id", "alignment_group", "median_residual_um", "p90_residual_um",
                            "added_offset_um", "dx_um", "dy_um"] if c in src]
    core = core_means(src[required + metadata], baseline_condition="baseline")
    keys = ["bundle_id", "sample_id", "core_id", "method", "condition"]
    flags = src.groupby(keys, dropna=False, observed=True)["_audit_excess"].any().reset_index()
    core = core.merge(flags, on=keys, how="left", validate="one_to_one")
    if "alignment_group" not in core:
        core["alignment_group"] = "unspecified"
    core["alignment_group"] = core["alignment_group"].fillna("unspecified").astype(str)
    methods = [m for m in ["bin2cell", "smurf_lite", "proseg"] if m in set(core["method"])]
    methods += sorted(set(core["method"]) - set(methods))
    present_groups = set(core["alignment_group"])
    groups = [g for g in ["good", "poor"] if g in present_groups]
    groups += sorted(present_groups - set(groups))
    units = ["bundle_id", "sample_id", "core_id"]
    method_coverage = core.groupby(units, dropna=False, observed=True)["method"].nunique()
    n_common = int(method_coverage.eq(len(methods)).sum())
    if n_common != len(method_coverage):
        raise ValueError("Use the same complete core set for every method; do not silently drop unmatched cores.")
    if core.groupby(units, dropna=False, observed=True)["alignment_group"].nunique().gt(1).any():
        raise ValueError("A core's alignment group differs between methods.")

    colors = {"bin2cell": "#2563eb", "smurf_lite": "#d97706", "proseg": "#0f766e"}
    extras = ["#9333ea", "#be185d", "#64748b"]
    for i, method in enumerate(methods):
        colors.setdefault(method, extras[i % len(extras)])
    n_sources = len(diag)
    n_flagged = int(diag["_audit_excess"].sum())
    fig, axes = plt.subplots(2, 2, figsize=(14, 10.6))
    fig.set_facecolor("white")
    shifts = np.linspace(-0.24, 0.24, len(methods)) if len(methods) > 1 else [0.0]
    panels = [
        ("A  Gene-count error", metrics[0], "L1 error / reference counts; lower is better"),
        ("B  Gene-count recovery", metrics[1], "Overlap / reference counts"),
        ("C  Gene-count precision", metrics[2], "Overlap / predicted counts"),
        ("D  Matched seed retention", metrics[3], "Fraction of frozen matched seeds present"),
    ]
    undefined = []
    for ax, (heading, metric, ylabel) in zip(axes.flat, panels):
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", color="#dfe6ef", linewidth=0.7)
        ax.set_axisbelow(True)
        ax.set_title(heading, loc="left", fontsize=14, weight="bold", pad=10)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.tick_params(labelsize=11)
        for j, method in enumerate(methods):
            for i, group in enumerate(groups):
                chosen = core.loc[core["method"].eq(method) & core["alignment_group"].eq(group)].sort_values(units)
                vals = pd.to_numeric(chosen[metric], errors="raise").to_numpy(float)
                finite = np.isfinite(vals)
                if (~finite).any():
                    undefined.append({"method": method, "alignment_group": group, "metric": metric,
                                      "n_undefined_core_metrics": int((~finite).sum())})
                chosen = chosen.loc[finite]
                vals = vals[finite]
                if not len(vals):
                    continue
                jitter = np.linspace(-0.045, 0.045, len(vals)) if len(vals) > 1 else np.array([0.0])
                xx = i + shifts[j] + jitter
                ax.scatter(xx, vals, color=colors[method], s=37, alpha=0.65, linewidths=0.8)
                marked = chosen["_audit_excess"].to_numpy(bool)
                if marked.any():
                    ax.scatter(xx[marked], vals[marked], marker="^", s=112, facecolors="none",
                               edgecolors="#b91c1c", linewidths=1.7, zorder=5)
                median = float(np.median(vals))
                ax.plot([i + shifts[j] - 0.07, i + shifts[j] + 0.07], [median, median],
                        color=colors[method], linewidth=3, solid_capstyle="butt", zorder=6)
        ax.set_xticks(range(len(groups)), groups)
        ax.set_xlim(-0.5, max(0.5, len(groups) - 0.5))
        # Autoscaling deliberately retains values outside [0, 1] and all outliers.
        ax.margins(y=0.12)
        if ax.lines or ax.collections:
            ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        else:
            ax.text(0.5, 0.5, "No defined values", transform=ax.transAxes, ha="center")
    headline = str(title).strip()
    if not headline.upper().startswith("EXPLORATORY"):
        headline = "EXPLORATORY · " + headline
    fig.suptitle(textwrap.fill(headline, width=83), x=0.06, y=0.982, ha="left", fontsize=19, weight="bold")
    fig.text(0.06, 0.920,
             f"{n_common} common cores · {n_sources} included run sources · {n_flagged} with bundle-level gene-count excess",
             fontsize=12, color="#475569")
    handles = [Line2D([0], [0], marker="o", linestyle="", color=colors[m], markersize=7, label=m) for m in methods]
    handles.append(Line2D([0], [0], marker="^", linestyle="", color="#b91c1c", markerfacecolor="none",
                          markeredgewidth=1.7, markersize=9, label="Source bundle has gene-count excess"))
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.895), ncol=2, frameon=False, fontsize=11)
    footnote = (
        "Dots: core scores averaged over technical repeats; bars: medians across those cores. Missing output cells remain zero.\n"
        "Triangles flag all cores from a source bundle with excess in any included repeat; they do not locate the excess within cores.\n"
        "Native counts are unchanged. Reference agreement is conditional on shared frozen seed matches; it is not molecule-identity accuracy.\n"
        "Cores within one specimen are not independent biological replicates."
    )
    if undefined:
        n_undefined = sum(d["n_undefined_core_metrics"] for d in undefined)
        footnote += f" {n_undefined} undefined core/metric values are omitted from the relevant panel only."
    fig.text(0.06, 0.025, footnote, fontsize=9, color="#475569", va="bottom", linespacing=1.45)
    fig.subplots_adjust(left=0.09, right=0.98, top=0.79, bottom=0.14, hspace=0.37, wspace=0.30)
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    png, pdf = output.with_suffix(".png"), output.with_suffix(".pdf")
    fig.savefig(png, dpi=200, facecolor="white")
    fig.savefig(pdf, facecolor="white")
    plt.close(fig)
    return {"png": str(png), "pdf": str(pdf), "n_common_cores": n_common,
            "n_run_sources": n_sources, "n_sources_with_gene_excess": n_flagged,
            "undefined_core_metrics": undefined}

In [ ]:
from IPython.display import Image
specimen = next(iter(config["samples"]))
plot_info = plot_exploratory_agreement(
    tables, audit, REPORT / "native_count_agreement.png",
    title=f"{specimen} · native-output reference agreement",
)
write_json(REPORT / "native_count_plot_audit.json", plot_info)
display(Image(filename=plot_info["png"]))

# Additional original panels: residual versus error, counts, recorded runner time and memory.
# Source triangles are in native_count_agreement; this second figure uses the original plotter.
flagged_sources = int(audit.source_bundle_has_gene_excess.sum())
plot_benchmark_report(tables, REPORT / "benchmark_summary.png",
    title=f"EXPLORATORY · {specimen} · {flagged_sources} sources with count excess")
display(Image(filename=str(REPORT / "benchmark_summary.png")))
print("Exploratory report:", REPORT)
print("Main per-core metrics:", REPORT / "all_core_metrics.csv")
print("Bundle/run count audit:", REPORT / "fresh_run_audit.csv")
print("Exact affected genes:", REPORT / "gene_excess_detail.csv")
print("Original recorded runtimes:", REPORT / "all_runtime_audits.csv")

## Reading and sharing the results
The main comparison retains every selected source and its native counts. The cause of SMURF excess is unresolved; small net excess cannot certify all other assignments. Read L1, recovery, precision, cosine and reference/seed coverage together. These scores describe agreement with Xenium conditional on frozen matches, not absolute truth or individual-molecule identity.

For review, share fresh_run_audit.csv, gene_excess_detail.csv, all_core_metrics.csv and native_count_agreement.png. all_runtime_audits.csv preserves the original recorded times. This count-excess warning does not require a return to 04A–04C.